# Template Eksperimen MSML
Ini adalah notebook untuk melakukan eksperimen EDA dan Preprocessing secara manual sesuai dengan Kriteria 1.

# 1. Perkenalan Dataset

Dataset yang digunakan dalam eksperimen ini adalah **Student Performance Dataset** yang diperoleh dari UCI Machine Learning Repository.

**Deskripsi Dataset**:
Dataset ini berisi data pencapaian akademik siswa pendidikan menengah dari dua sekolah di Portugal. Fitur-fitur (kolom) pada data ini mencakup nilai siswa (G1, G2, dan target akhir G3), data demografis (umur, jenis kelamin, alamat rumah), lingkungan sosial dan keluarga (pendidikan orang tua, ukuran keluarga, status kohabitasi), serta gaya hidup dan kebiasaan yang berkaitan dengan sekolah (waktu belajar tambahan, jumlah absen, waktu luang, konsumsi alkohol, dsb).

**Tujuan / Problem Statement**:
Tujuan dari pemodelan ini adalah memprediksi nilai akademik akhir siswa (kolom G3) berdasarkan fitur sosial, demografis, dan belajarnya. Dengan demikian, model Machine Learning ini dapat membantu pihak sekolah untuk mendeteksi dini siswa-siswa yang berisiko gagal atau mendapat nilai rendah, sehingga intervensi/bantuan belajar yang tepat bisa segera diberikan.

# 2. Import Library

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
import os
import warnings
warnings.filterwarnings('ignore')

# 3. Memuat Dataset

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.
Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.
Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

In [ ]:
# Memuat dataset
train_df = pd.read_csv('../student_raw/train.csv')
test_df = pd.read_csv('../student_raw/test.csv')
train_df.head()

# 4. Exploratory Data Analysis (EDA)

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.
Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

In [ ]:
# Mengecek missing values
train_df.isnull().sum()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(train_df['G3'], bins=20, kde=True)
plt.title('Distribusi Nilai Akhir (G3)')
plt.show()

# 5. Data Preprocessing

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.
Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.
Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

In [ ]:
TARGET_COL = 'Label Kelulusan (0/1)'

if 'G3' in train_df.columns:
    train_df[TARGET_COL] = train_df['G3'].apply(lambda x: 1 if x >= 10 else 0)
    test_df[TARGET_COL] = test_df['G3'].apply(lambda x: 1 if x >= 10 else 0)

cols_to_drop = ['G1', 'G2', 'G3']
train_df = train_df.drop(columns=[c for c in cols_to_drop if c in train_df.columns])
test_df = test_df.drop(columns=[c for c in cols_to_drop if c in test_df.columns])

In [ ]:
cat_cols = train_df.select_dtypes(include=['object']).columns
for col in cat_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = test_df[col].apply(lambda x: x if x in le.classes_ else train_df[col].mode()[0])
    le_dict = dict(zip(le.classes_, le.transform(le.classes_)))
    test_df[col] = test_df[col].map(le_dict).fillna(train_df[col].mode()[0]).astype(int)

In [ ]:
num_cols = train_df.select_dtypes(include=['int64', 'float64']).columns
num_cols = [c for c in num_cols if c != TARGET_COL]

scaler = StandardScaler()
train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
test_df[num_cols] = scaler.transform(test_df[num_cols])

In [ ]:
train_df.head()

In [ ]:
os.makedirs('../student_preprocessing', exist_ok=True)
train_df.to_csv('../student_preprocessing/train.csv', index=False)
test_df.to_csv('../student_preprocessing/test.csv', index=False)
print('Data bersih berhasil disimpan ke ../student_preprocessing')